<a href="https://colab.research.google.com/github/iman-ch/contract-conflict-checker/blob/main/02_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notebook Overview: Find best general contradition model to train, and build reusable tools for rest of project

## 1. Setup

### **1.1** install dependecies
  - transformers (pretrained models, tokenizers)
  - datasets (mnli)
  - mlflow (to record experiment settings)
  - scikit-learn

In [ ]:
!pip install -q transformers datasets mlflow scikit-learn
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### **1.2** imports and configuration

In [ ]:
import os
import time
import torch
import mlflow
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding)
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

device = torch.device("cuda")
print(torch.cuda.is_available())

PROJ_DIR = "/content/drive/MyDrive/doc-contradiction-detector"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri(f"file:{PROJ_DIR}/mlruns")
mlflow.set_experiment("nli-distilbert")

True


<Experiment: artifact_location='file:///content/drive/MyDrive/doc-contradiction-detector/mlruns/647002999600830066', creation_time=1790968466273, effective_trace_archival_retention=None, experiment_id='647002999600830066', last_update_time=1790968466273, lifecycle_stage='active', name='nli-distilbert', tags={}, trace_location=None, workspace='default'>

### **1.3** load dataset, tokenizer

In [ ]:
mnli = load_dataset("nyu-mll/multi_nli")
label_names = mnli["train"].features["label"].names
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(label_names)

README.md:   0%|          | 0.00/8.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  214MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation_matched-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 4.94MB            

data/validation_matched-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

['entailment', 'neutral', 'contradiction']


all similar to notebook one

## 2. Dynamic Padding

### **2.1** Tokenization Fix

in 01_baseline section 3.2, cell output ended in long [PAD] runs since every pair was padded to 128 tokens.

Fix: pad batch to length of its own largest pair using collator (combine rows into one batch)

In [ ]:
def tokenize(batch):
  return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, max_length=128)

keep = ["input_ids", "attention_mask", "label"]
demo_ds = mnli["train"].shuffle(seed=42).select(range(20000)).map(tokenize, batched=True)
demo_ds = demo_ds.remove_columns([c for c in demo_ds.column_names if c not in keep])

collator = DataCollatorWithPadding(tokenizer)
demo_loader = DataLoader(demo_ds, batch_size=32, shuffle=True, collate_fn=collator)

for i, batch in enumerate(demo_loader):
  print(batch["input_ids"].shape)
  if i == 4:
    break
print(batch.keys())

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

torch.Size([32, 77])
torch.Size([32, 84])
torch.Size([32, 82])
torch.Size([32, 73])
torch.Size([32, 128])
KeysView({'input_ids': tensor([[ 101, 2138, 1045,  ...,    0,    0,    0],
        [ 101, 2057, 1005,  ...,    0,    0,    0],
        [ 101, 2054, 2065,  ...,    0,    0,    0],
        ...,
        [ 101, 1000, 5622,  ...,    0,    0,    0],
        [ 101, 2648, 1996,  ...,    0,    0,    0],
        [ 101, 1996, 8312,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([0, 2, 1, 2, 2, 1, 2, 0, 2, 0, 2, 0, 2, 2, 0, 2, 1, 0, 2, 0, 2, 0, 0, 1,
        0, 2, 1, 2, 1, 1, 2, 1])})


before, batches were padded to 128, bc transformers need everything to be same length. data collator pads each batch only to longest sequence to cut training time.

## 3. Mixed Precision

Most of the forward pass runs in 16-bit numbers (float16) instead of 32-bit (float32). GPUs compute 16-bit math much faster, roughly halving training time with almost no accuracy loss.

**Problem:** float16 can't store very small numbers. Tiny gradients round down to zero (underflow), and those weights stop learning.

**Fix, GradScaler:** multiplies the loss by a big number before backprop so gradients stay large enough to store, then shrinks them back before the optimizer step.

**Code pattern** (used inside `train_one_epoch` in Section 4):
- `torch.autocast(...)` wraps the forward pass and loss: runs them in mixed precision
- `scaler.scale(loss).backward()` replaces `loss.backward()`
- `scaler.step(optimizer)` replaces `optimizer.step()`
- `scaler.update()` adjusts the scale factor each step

## 4. Training, Evaluation Functions

### **4.1** train one epoch

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, scaler):
  model.train()
  total_loss = 0
  for batch in tqdm(loader, desc="train", leave=False):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad() # zero
    with torch.autocast(device_type="cuda", dtype=torch.float16):
      logits = model(input_ids=input_ids, attention_mask=attention_mask).logits  # fwd
      loss = loss_fn(logits, labels) # loss
    scaler.scale(loss).backward() # bkwd
    scaler.step(optimizer) # step
    scaler.update()

    total_loss += loss.item()
  return total_loss / len(loader)

### **4.2** evaluate

In [ ]:
def evaluate(model, loader):
  model.eval()
  all_preds, all_labels = [], []
  with torch.no_grad():
    for batch in loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      with torch.autocast(device_type="cuda", dtype=torch.float16):
          logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
      all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
      all_labels.extend(batch["labels"].tolist())
  acc = accuracy_score(all_labels, all_preds)
  f1s = f1_score(all_labels, all_preds, average=None)

  return acc, f1s

## 5. Experiment Tracking (MLFlow)

MLflow automatically records every training run's settings and results, so runs can be compared side by side instead of relying on memory or scrolling through printed output.

**Key terms:**
- **Experiment:** a named group of related runs (`nli-distilbert`)
- **Run:** one training attempt (e.g. lr 3e-5, 100K pairs)
- **Parameter:** a setting chosen before training, logged once (lr, max_length, n_train, epochs, batch_size, seed)
- **Metric:** a result produced by training, logged every epoch (train_loss, val_accuracy, per-class F1, minutes)


## 6. Experiment Function

### **6.1** run experiment



In [ ]:
def run_experiment(lr, max_length, n_train, epochs=2, batch_size=32, seed=42):
  start = time.time()
  torch.manual_seed(seed)

  # 1. data (notebook 1: 2.4, 3, 4)
  def tok(batch):
    return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, max_length=max_length)

  keep = ["input_ids", "attention_mask", "label"]
  train_ds = mnli["train"].shuffle(seed=seed).select(range(n_train)).map(tok, batched=True)
  train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
  val_ds = mnli["validation_matched"].map(tok, batched=True)
  val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep])

  collator = DataCollatorWithPadding(tokenizer)
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collator)
  val_loader = DataLoader(val_ds, batch_size=128, collate_fn=collator)

  # 2. fresh model and training tools (notebook 1: 5)
  model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3).to(device)
  optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
  loss_fn = torch.nn.CrossEntropyLoss()
  scaler = torch.amp.GradScaler("cuda")

  # 3. train, evaluate, log (notebook 1: 6.1, plus MLflow)
  run_name = f"lr{lr}_len{max_length}_n{n_train // 1000}k"
  with mlflow.start_run(run_name=run_name):
    mlflow.log_params({"lr": lr, "max_length": max_length, "n_train": n_train, "epochs": epochs, "batch_size": batch_size, "seed": seed})
    for epoch in range(epochs):
      train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler)
      acc, f1s = evaluate(model, val_loader)
      mlflow.log_metric("train_loss", train_loss, step=epoch)
      mlflow.log_metric("val_accuracy", acc, step=epoch)
      for name, f in zip(label_names, f1s):
          mlflow.log_metric(f"f1_{name}", f, step=epoch)
      print(f"{run_name} | epoch {epoch + 1} | loss {train_loss:.4f} | acc {acc:.2%}")
    minutes = (time.time() - start) / 60
    mlflow.log_metric("minutes", minutes)

  # 4. Save (notebook 1: 6.2)
  model.save_pretrained(f"{PROJ_DIR}/models/{run_name}")
  tokenizer.save_pretrained(f"{PROJ_DIR}/models/{run_name}")
  print(f"Done in {minutes:.1f} min")
  return acc

## 7. Experiment Runs

### **7.1** smoke check

In [ ]:
run_experiment(lr=2e-5, max_length=128, n_train=2000, epochs=1)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

train:   0%|          | 0/63 [00:00<?, ?it/s]

lr2e-05_len128_n2k | epoch 1 | loss 1.0924 | acc 48.89%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 0.5 min


0.48894549159449824

### **7.2** baseline check

In [ ]:
run_experiment(lr=2e-5, max_length=128, n_train=20000, epochs=2)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


train:   0%|          | 0/625 [00:00<?, ?it/s]

lr2e-05_len128_n20k | epoch 1 | loss 0.8467 | acc 69.63%


train:   0%|          | 0/625 [00:00<?, ?it/s]

lr2e-05_len128_n20k | epoch 2 | loss 0.6083 | acc 70.80%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 2.2 min


0.707997962302598

### **7.3** learning rate

In [ ]:
results = {}
for lr in [2e-5, 3e-5, 5e-5]:
  results[lr] = run_experiment(lr=lr, max_length=128, n_train=50000)
print(results)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


train:   0%|          | 0/1563 [00:00<?, ?it/s]

lr2e-05_len128_n50k | epoch 1 | loss 0.7588 | acc 73.80%


...

lr5e-05_len128_n50k | epoch 1 | loss 0.7558 | acc 73.66%


train:   0%|          | 0/1563 [00:00<?, ?it/s]

lr5e-05_len128_n50k | epoch 2 | loss 0.5061 | acc 74.69%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 5.1 min
{2e-05: 0.7519103413143148, 3e-05: 0.7550687722873153, 5e-05: 0.7469179826795721}


### **7.4** sequence length


In [ ]:
run_experiment(lr=3e-5, max_length=256, n_train=50000)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


train:   0%|          | 0/1563 [00:00<?, ?it/s]

...

train:   0%|          | 0/1563 [00:00<?, ?it/s]

lr3e-05_len256_n50k | epoch 2 | loss 0.5124 | acc 74.71%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 5.9 min


0.7471217524197656

kept lr 3e-5 (best from 7.3); tested max_length 256 vs. 128. 256 scored 74.7% vs. 75.5% and ran slower

few MultiNLI pairs exceed 128 tokens, so the longer limit adds nothing; 128 is kept

### **7.5** more training data

In [ ]:
run_experiment(lr=3e-5, max_length=128, n_train=100000)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


train:   0%|          | 0/3125 [00:00<?, ?it/s]

lr3e-05_len128_n100k | epoch 1 | loss 0.6925 | acc 76.69%


train:   0%|          | 0/3125 [00:00<?, ?it/s]

lr3e-05_len128_n100k | epoch 2 | loss 0.4741 | acc 77.47%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 10.3 min


0.7747325522159959

In [ ]:
BEST_GENERAL = f"{PROJ_DIR}/models/lr3e-05_len128_n100k"

## 8. Results

In [ ]:
runs = mlflow.search_runs(experiment_names=["nli-distilbert"])

cols = {
    "tags.mlflow.runName": "Run",
    "params.lr": "lr",
    "params.max_length": "max_length",
    "params.n_train": "Train pairs",
    "metrics.val_accuracy": "Val accuracy",
    "metrics.f1_contradiction": "Contradiction F1",
    "metrics.minutes": "Minutes",
}
results = runs[list(cols)].rename(columns=cols)
results = results[results["Train pairs"] != "2000"]          # drop the smoke test
results = results.sort_values("Val accuracy", ascending=False).round(3)
results

,Run,lr,max_length,Train pairs,Val accuracy,Contradiction F1,Minutes
0,lr3e-05_len128_n100k,3e-05,128,100000,0.775,0.787,10.321
3,lr3e-05_len128_n50k,3e-05,128,50000,0.755,0.763,5.082
4,lr2e-05_len128_n50k,2e-05,128,50000,0.752,0.756,5.294
1,lr3e-05_len256_n50k,3e-05,256,50000,0.747,0.756,5.932
2,lr5e-05_len128_n50k,5e-05,128,50000,0.747,0.755,5.081
5,lr2e-05_len128_n20k,2e-05,128,20000,0.708,0.710,2.209


**Notebook 1 baseline** (not logged in MLflow): 71.5% on 2K validation pairs, ~10 min.

**What I learned**
- **Training data mattered most.** Going from 20K -> 50K -> 100K pairs raised accuracy from 70.8% -> 75.5% -> 77.5%. Nothing else came close.
- **Learning rate mattered a little.** 3e-5 was best, but only 0.3 points ahead of 2e-5. 5e-5 was worst, suggesting it's too large for fine-tuning.
- **max_length 256 added nothing.** Few MultiNLI pairs exceed 128 tokens, so 128 is kept.
- **The optimized pipeline is 4–5× faster.** Dynamic padding and mixed precision cut the 20K baseline from ~10 to 2.2 minutes, with accuracy within a point of notebook 1.
- **Contradiction F1 improved with data too.** It rose from 0.710 (20K) to 0.787 (100K), and it's the highest of the three classes.

**Best model:** `models/lr3e-05_len128_n100k` (77.5%), the starting point for notebook 3.